# 模块说明    
本模块用于演示money_flow各项功能

In [6]:
import sys
import os
# 获取当前Notebook的绝对路径
notebook_dir = os.path.abspath('')
# 项目根目录是Notebook所在目录的父目录
project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))
# 将项目根目录添加到sys.path
if project_root not in sys.path:
    sys.path.insert(0, project_root)
# 确认路径是否正确
print("当前工作目录:", os.getcwd())
print("项目根目录:", project_root)
print("sys.path:", sys.path)


当前工作目录: c:\Users\GHUIQ\repos\MyFunds\jupyterbook\akshare
项目根目录: c:\Users\GHUIQ\repos\MyFunds
sys.path: ['c:\\Users\\GHUIQ\\repos\\MyFunds', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\python311.zip', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\DLLs', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\Lib', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda', '', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\Lib\\site-packages', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\Lib\\site-packages\\win32', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\GHUIQ\\repos\\MyFunds\\.conda\\Lib\\site-packages\\Pythonwin']


In [7]:
import numpy as np
import pandas as pd
import tushare as ts
import akshare as ak
import sqlalchemy
from scipy import stats
from datetime import datetime
#from apt.vendor.akshare.base import base as base
#from apt.vendor.akshare.base import stock as stock
from apt.vendor.akshare.data import data as akdata
from apt.vendor.akshare.money_flow import money_flow as money_flow
from sqlalchemy.types import NVARCHAR , Float, Integer , Date
from apt.vendor.tspro.security import security as sec

In [8]:
# 演示使用money_flow加载日线数据
money = money_flow()
money.start_date = datetime(2025, 1, 1)
money.end_date = datetime.now()
money.ktype = '1d'
money.code = '300100.SZ'
df_day = money.get_k_data()
print(df_day)

#这里的复权可能有些问题

          code        date       open      close       high        low  \
0    300100.SZ  2025-01-02  19.926220  20.269036  21.076084  19.140598   
1    300100.SZ  2025-01-03  20.304746  19.004900  20.854681  18.890627   
2    300100.SZ  2025-01-06  18.740645  18.433539  19.297722  18.162142   
3    300100.SZ  2025-01-07  18.490675  19.176308  19.212018  18.354976   
4    300100.SZ  2025-01-08  18.926338  20.226184  20.576143  18.454965   
..         ...         ...        ...        ...        ...        ...   
168  300100.SZ  2025-09-10  45.500000  44.850000  45.850000  44.680000   
169  300100.SZ  2025-09-11  44.580000  45.980000  45.990000  44.140000   
170  300100.SZ  2025-09-12  45.990000  45.290000  46.770000  45.280000   
171  300100.SZ  2025-09-15  45.230000  45.400000  46.700000  45.120000   
172  300100.SZ  2025-09-16  45.410000  48.690000  49.490000  45.310000   

         volume         money  factor  
0    31928700.0  9.095103e+08   3.631  
1    27480403.0  7.609520e+08  

In [ ]:
# 演示计算基于分时线的资金流向
df_money_flow_min = money.calculate_money_flow_min()
print("----基于分时线的资金流向数据:")
print(f"分时数据总行数: {len(df_money_flow_min)}")
print(df_money_flow_min.head())

# 演示聚合后的资金流向（日线）
# 确保日期列是datetime类型
df_money_flow_min['date'] = pd.to_datetime(df_money_flow_min['date'])
# 提取日期部分用于分组
df_money_flow_min['trade_date'] = df_money_flow_min['date'].dt.date

# 按股票代码和日期分组聚合
df_money_flow_daily = df_money_flow_min.groupby(['code', 'trade_date']).agg({
    '净资金流向': 'sum',
    '波动比率': 'mean',  # 波动比率取平均值
    'volume': 'sum',     # 成交量求和
    'money': 'sum'       # 成交金额求和
}).reset_index()

# 重命名列以保持一致性
df_money_flow_daily.rename(columns={'trade_date': 'date'}, inplace=True)

print("----聚合后的资金流向（日线）:")
print(f"日线数据行数: {len(df_money_flow_daily)}")
print(df_money_flow_daily)

----基于分时线的资金流向数据:
            code                date       open      close       high  \
0      300100.SZ 2025-01-02 09:30:00  19.926220  19.926220  19.926220   
1      300100.SZ 2025-01-02 09:31:00  19.840515  19.554835  19.861941   
2      300100.SZ 2025-01-02 09:32:00  19.554835  19.312006  19.554835   
3      300100.SZ 2025-01-02 09:33:00  19.404852  19.447705  19.447705   
4      300100.SZ 2025-01-02 09:34:00  19.419137  19.226302  19.447705   
...          ...                 ...        ...        ...        ...   
41688  300100.SZ 2025-09-16 14:56:00  48.680000  48.690000  48.700000   
41689  300100.SZ 2025-09-16 14:57:00  48.700000  48.690000  48.700000   
41690  300100.SZ 2025-09-16 14:58:00  48.690000  48.690000  48.690000   
41691  300100.SZ 2025-09-16 14:59:00  48.690000  48.690000  48.690000   
41692  300100.SZ 2025-09-16 15:00:00  48.690000  48.690000  48.690000   

             low    volume       money  factor    波动比率        净资金流向  
0      19.926220  103200.0   287928

In [ ]:
# 删除这个重复且有错误的单元格，因为它包含语法错误且与上面的逻辑重复
# 上面已经完成了正确的聚合操作
print("该单元格已被清理，聚合逻辑已在上面完成")

NameError: name '_money_flow_min' is not defined

In [ ]:
# 演示获取基于分时线的资金流向数据
# 获取分时线的资金流向数据
money = money_flow()
money.code = '300100.SZ'  # 使用与前面相同的股票代码
# 定义一个函数来获取指定日期的资金流向数据
def get_money_flow_for_date(date):
    try:
        money.start_date = date
        money.end_date = date
        result = money.calculate_money_flow_min()
        return pd.Series([result['波动比率'], result['净资金流向']], index=['波动比率', '净资金流向'])
    except Exception as e:
        print(f"处理日期 {date} 时出错: {e}")
        return pd.Series([np.nan, np.nan], index=['波动比率', '净资金流向'])

# 将日期列转换为datetime对象
df['date'] = pd.to_datetime(df['date'])

# 使用apply应用函数到每一行
flow_data = df['date'].apply(get_money_flow_for_date)

# 将结果合并到原始DataFrame
df[['波动比率', '净资金流向']] = flow_data

# 显示添加了新列的数据
print(df[['code', 'date', 'close', '波动比率', '净资金流向']].tail())
